# 03 - ECC Application (GPU)

In [65]:
import subprocess, sys

# Try to bring up PyCUDA. If there is no GPU (or it will not build),
# fall back to a pure-Python path so the notebook still runs end to end.
GPU_AVAILABLE = False
try:
    import pycuda.autoinit
    import pycuda.driver as cuda
    from pycuda.compiler import SourceModule
    GPU_AVAILABLE = True
except Exception:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pycuda", "-q"])
        import pycuda.autoinit
        import pycuda.driver as cuda
        from pycuda.compiler import SourceModule
        GPU_AVAILABLE = True
    except Exception as e:
        print(f"No usable GPU ({e}). Using CPU fallback, results are still correct.")

import numpy as np

if GPU_AVAILABLE:
    dev = cuda.Device(0)
    print(f"GPU : {dev.name()}")
    print(f"Mem : {dev.total_memory() / 1e9:.1f} GB")
    print(f"CC  : {dev.compute_capability()}")
else:
    print("Running on CPU fallback.")

GPU : NVIDIA GeForce RTX 5070
Mem : 12.8 GB
CC  : (12, 0)


In [66]:

# for matt local machine, comment out if using datahub 

msvc_bin = r"C:\Program Files\Microsoft Visual Studio\2022\Community\VC\Tools\MSVC\14.44.35207\bin\Hostx64\x64"
os.environ["PATH"] = msvc_bin + ";" + os.environ["PATH"]


In [67]:
# NIST P-256 prime:  P = 2^256 - 2^224 + 2^192 + 2^96 - 1  (FIPS 186-4)
P = 0xFFFFFFFF00000001000000000000000000000000FFFFFFFFFFFFFFFFFFFFFFFF

# R^2 mod P, with R = 2^256. Used to push operands into Montgomery form.
# Computed once on the host and uploaded to the device at startup.
R2 = pow(2, 512, P)

print(f"P    = {hex(P)}")
print(f"R2   = {hex(R2)}")
print(f"bits = {P.bit_length()}")

# numpy dtype matching the CUDA struct: 4 x uint64, limb[0] = least significant.
u256_t = np.dtype([("limb", np.uint64, (4,))], align=True)
point_t = np.dtype([
    ("x", u256_t),
    ("y", u256_t),
    ("infinity", np.int32),
], align=True)

print("u256_t size:", u256_t.itemsize)
print("point_t size:", point_t.itemsize)
assert u256_t.itemsize == 32
assert point_t.itemsize == 72

MASK64 = 0xFFFFFFFFFFFFFFFF

def to_arr(nums):
    """List of Python ints -> numpy u256 array (little-endian limbs)."""
    a = np.zeros(len(nums), dtype=u256_t)
    for idx, n in enumerate(nums):
        for i in range(4):
            a["limb"][idx, i] = (n >> (64 * i)) & MASK64
    return a

def to_list(a):
    """numpy u256 array -> list of Python ints."""
    out = []
    for idx in range(len(a)):
        v = 0
        for i in range(4):
            v |= int(a["limb"][idx, i]) << (64 * i)
        out.append(v)
    return out

print("Helpers defined.")

P    = 0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff
R2   = 0x4fffffffdfffffffffffffffefffffffbffffffff0000000000000003
bits = 256
u256_t size: 32
point_t size: 72
Helpers defined.


In [68]:
from kernels import KERNEL_SRC

ECC_KERNEL_SRC = KERNEL_SRC + r"""

// ---------- ECC over NIST P-256 ----------
// Uses Jacobian coordinates internally for faster scalar multiplication.
// Affine point:  (x, y)
// Jacobian point: (X, Y, Z), where affine x = X/Z^2, y = Y/Z^3

typedef struct {
    u256 x;
    u256 y;
    int infinity;
} ec_point;

typedef struct {
    u256 X;
    u256 Y;
    u256 Z;
    int infinity;
} jac_point;

// ---------- constants/helpers ----------

__device__ u256 make_zero() {
    u256 z;
    z.limb[0] = 0ULL;
    z.limb[1] = 0ULL;
    z.limb[2] = 0ULL;
    z.limb[3] = 0ULL;
    return z;
}

__device__ u256 make_one_u256() {
    u256 one;
    one.limb[0] = 1ULL;
    one.limb[1] = 0ULL;
    one.limb[2] = 0ULL;
    one.limb[3] = 0ULL;
    return one;
}

__device__ u256 make_three() {
    u256 three;
    three.limb[0] = 3ULL;
    three.limb[1] = 0ULL;
    three.limb[2] = 0ULL;
    three.limb[3] = 0ULL;
    return three;
}

__device__ u256 make_p_minus_2() {
    u256 e;
    e.limb[0] = 0xFFFFFFFFFFFFFFFDULL;
    e.limb[1] = 0x00000000FFFFFFFFULL;
    e.limb[2] = 0x0000000000000000ULL;
    e.limb[3] = 0xFFFFFFFF00000001ULL;
    return e;
}

__device__ int is_zero256(u256 a) {
    return (
        a.limb[0] == 0ULL &&
        a.limb[1] == 0ULL &&
        a.limb[2] == 0ULL &&
        a.limb[3] == 0ULL
    );
}

__device__ u256 d_mod_inv(u256 a) {
    return d_mod_exp(a, make_p_minus_2());
}

__device__ ec_point make_affine_inf() {
    ec_point r;
    r.x = make_zero();
    r.y = make_zero();
    r.infinity = 1;
    return r;
}

__device__ jac_point make_jac_inf() {
    jac_point r;
    r.X = make_zero();
    r.Y = make_zero();
    r.Z = make_zero();
    r.infinity = 1;
    return r;
}

__device__ jac_point affine_to_jac(ec_point P) {
    if (P.infinity) {
        return make_jac_inf();
    }

    jac_point R;
    R.X = P.x;
    R.Y = P.y;
    R.Z = make_one_u256();
    R.infinity = 0;
    return R;
}

__device__ ec_point jac_to_affine(jac_point P) {
    if (P.infinity || is_zero256(P.Z)) {
        return make_affine_inf();
    }

    u256 z_inv = d_mod_inv(P.Z);
    u256 z_inv2 = d_mod_mul(z_inv, z_inv);
    u256 z_inv3 = d_mod_mul(z_inv2, z_inv);

    ec_point R;
    R.x = d_mod_mul(P.X, z_inv2);
    R.y = d_mod_mul(P.Y, z_inv3);
    R.infinity = 0;
    return R;
}

// ---------- Jacobian point doubling ----------
// Formula optimized for a = -3.
// alpha = 3 * (X - Z^2) * (X + Z^2)
// beta  = X * Y^2
// X3    = alpha^2 - 8*beta
// Y3    = alpha*(4*beta - X3) - 8*Y^4
// Z3    = (Y + Z)^2 - Y^2 - Z^2

__device__ jac_point d_jac_double(jac_point P) {
    if (P.infinity || is_zero256(P.Y)) {
        return make_jac_inf();
    }

    u256 three = make_three();

    u256 Z2 = d_mod_mul(P.Z, P.Z);
    u256 Y2 = d_mod_mul(P.Y, P.Y);
    u256 Y4 = d_mod_mul(Y2, Y2);

    u256 X_minus_Z2 = d_mod_sub(P.X, Z2);
    u256 X_plus_Z2 = d_mod_add(P.X, Z2);

    u256 alpha = d_mod_mul(three, d_mod_mul(X_minus_Z2, X_plus_Z2));
    u256 beta = d_mod_mul(P.X, Y2);

    u256 four_beta = d_mod_add(beta, beta);
    four_beta = d_mod_add(four_beta, four_beta);

    u256 eight_beta = d_mod_add(four_beta, four_beta);

    u256 alpha2 = d_mod_mul(alpha, alpha);
    u256 X3 = d_mod_sub(alpha2, eight_beta);

    u256 eight_Y4 = d_mod_add(Y4, Y4);
    eight_Y4 = d_mod_add(eight_Y4, eight_Y4);
    eight_Y4 = d_mod_add(eight_Y4, eight_Y4);

    u256 Y3 = d_mod_sub(
        d_mod_mul(alpha, d_mod_sub(four_beta, X3)),
        eight_Y4
    );

    u256 Y_plus_Z = d_mod_add(P.Y, P.Z);
    u256 Z3 = d_mod_sub(
        d_mod_sub(d_mod_mul(Y_plus_Z, Y_plus_Z), Y2),
        Z2
    );

    jac_point R;
    R.X = X3;
    R.Y = Y3;
    R.Z = Z3;
    R.infinity = 0;
    return R;
}

// ---------- Mixed Jacobian + affine addition ----------
// Adds Jacobian P + affine Q.
// Much faster than affine add because it avoids inverse.

__device__ jac_point d_jac_add_mixed(jac_point P, ec_point Q) {
    if (P.infinity) {
        return affine_to_jac(Q);
    }

    if (Q.infinity) {
        return P;
    }

    u256 Z1Z1 = d_mod_mul(P.Z, P.Z);
    u256 U2 = d_mod_mul(Q.x, Z1Z1);
    u256 S2 = d_mod_mul(Q.y, d_mod_mul(P.Z, Z1Z1));

    u256 H = d_mod_sub(U2, P.X);
    u256 R = d_mod_sub(S2, P.Y);

    if (is_zero256(H)) {
        if (is_zero256(R)) {
            return d_jac_double(P);
        } else {
            return make_jac_inf();
        }
    }

    u256 HH = d_mod_mul(H, H);
    u256 HHH = d_mod_mul(H, HH);
    u256 V = d_mod_mul(P.X, HH);

    u256 R2 = d_mod_mul(R, R);

    u256 two_V = d_mod_add(V, V);
    u256 X3 = d_mod_sub(d_mod_sub(R2, HHH), two_V);

    u256 Y3 = d_mod_sub(
        d_mod_mul(R, d_mod_sub(V, X3)),
        d_mod_mul(P.Y, HHH)
    );

    u256 Z3 = d_mod_mul(P.Z, H);

    jac_point Out;
    Out.X = X3;
    Out.Y = Y3;
    Out.Z = Z3;
    Out.infinity = 0;
    return Out;
}

// ---------- Affine wrappers ----------

__device__ ec_point d_point_double(ec_point P) {
    jac_point J = affine_to_jac(P);
    jac_point D = d_jac_double(J);
    return jac_to_affine(D);
}

__device__ ec_point d_point_add(ec_point P, ec_point Q) {
    jac_point J = affine_to_jac(P);
    jac_point S = d_jac_add_mixed(J, Q);
    return jac_to_affine(S);
}

// ---------- Scalar multiply ----------
// Left-to-right double-and-add.
// Only one inverse at the end.

__device__ ec_point d_scalar_mul(ec_point P, u256 k) {
    jac_point result = make_jac_inf();

    for (int i = 255; i >= 0; i--) {
        result = d_jac_double(result);

        int w = i / 64;
        int bit = i % 64;

        if ((k.limb[w] >> bit) & 1ULL) {
            result = d_jac_add_mixed(result, P);
        }
    }

    return jac_to_affine(result);
}

// ---------- kernels ----------

__global__ void kernel_point_add(ec_point *A, ec_point *B, ec_point *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n) {
        C[tid] = d_point_add(A[tid], B[tid]);
    }
}

__global__ void kernel_point_double(ec_point *A, ec_point *C, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n) {
        C[tid] = d_point_double(A[tid]);
    }
}

__global__ void kernel_scalar_mul(ec_point *Pnts, u256 *Ks, ec_point *Out, int n) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < n) {
        Out[tid] = d_scalar_mul(Pnts[tid], Ks[tid]);
    }
}

"""

print("ECC kernel source defined.")

ECC kernel source defined.


In [69]:
ecc_gpu = SourceModule(
    ECC_KERNEL_SRC,
    arch="sm_120",
    options=["--std=c++17"]
)

# Upload R2 again for this new module
R2_arr = np.array([(R2 >> (64 * i)) & MASK64 for i in range(4)], dtype=np.uint64)
r2_ptr, _ = ecc_gpu.get_global("R2_LIMBS")
cuda.memcpy_htod(r2_ptr, R2_arr)

k_point_add = ecc_gpu.get_function("kernel_point_add")
k_point_double = ecc_gpu.get_function("kernel_point_double")
k_scalar_mul = ecc_gpu.get_function("kernel_scalar_mul")

point_t = np.dtype([
    ("x", u256_t),
    ("y", u256_t),
    ("infinity", np.int32),
], align=True)

print("ECC kernels compiled.")

ECC kernels compiled.


C:\Users\matth\AppData\Local\Temp\ipykernel_31908\2103786616.py:1: UserWarning: The CUDA compiler succeeded, but said the following:
kernel.cu

  ecc_gpu = SourceModule(


In [70]:
def int_to_u256(x):
    u = np.zeros((), dtype=u256_t)
    for i in range(4):
        u["limb"][i] = (int(x) >> (64 * i)) & MASK64
    return u

def u256_to_int(x):
    return sum(int(x["limb"][i]) << (64 * i) for i in range(4))

def make_point(x, y, infinity=0):
    p = np.zeros((), dtype=point_t)
    p["x"] = int_to_u256(x)
    p["y"] = int_to_u256(y)
    p["infinity"] = int(infinity)
    return p

def point_to_tuple(p):
    if int(p["infinity"]) == 1:
        return None
    return (u256_to_int(p["x"]), u256_to_int(p["y"]))

def points_to_arr(points):
    arr = np.zeros(len(points), dtype=point_t)

    for i, pt in enumerate(points):
        if pt is None:
            arr[i] = make_point(0, 0, 1)
        else:
            arr[i] = make_point(pt[0], pt[1], 0)

    return arr

def scalars_to_arr(vals):
    arr = np.zeros(len(vals), dtype=u256_t)

    for i, v in enumerate(vals):
        arr[i] = int_to_u256(v)

    return arr

def arr_to_points(arr):
    return [point_to_tuple(arr[i]) for i in range(len(arr))]

In [71]:
from pathlib import Path
import sys

ROOT = Path.cwd().parents[1]   # from src/gpu -> mac-ecc
SRC = ROOT / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(SRC)

c:\Users\matth\Desktop\repos\mac-ecc\src


In [72]:
from cpu.ecc import (
    P,
    A_COEF,
    B_COEF,
    G,
    N,
    POINT_AT_INFINITY,
    point_add,
    point_double,
    scalar_multiply,
)

print("CPU ECC reference imported.")
print(f"P = {hex(P)}")
print(f"G = {G}")

CPU ECC reference imported.
P = 0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff
G = (48439561293906451759052585252797914202762949526041747995844080717082404635286, 36134250956749795798585127919587881956611106672985015071877198253568414405109)


In [73]:
MASK64 = (1 << 64) - 1
R = 1 << 256
R2 = (R * R) % P

BLOCK = 256

def grid(n):
    return (int(np.ceil(n / BLOCK)), 1)

def blk():
    return (BLOCK, 1, 1)

print(f"R2 = {hex(R2)}")

R2 = 0x4fffffffdfffffffffffffffefffffffbffffffff0000000000000003


In [74]:
print("Testing point_double_gpu...")
assert point_double_gpu(G) == point_double(G)
print("  point_double_gpu ok")

print("Testing point_add_gpu...")
G2 = point_double(G)
assert point_add_gpu(G, G2) == point_add(G, G2)
print("  point_add_gpu ok")

print("Testing scalar_mul_gpu...")
for k in [1, 2, 3, 5, 7, 11, 19]:
    got = scalar_mul_gpu(G, k)
    expected = scalar_multiply(k, G)
    assert got == expected, f"k={k} failed"

print("  scalar_mul_gpu ok")

Testing point_double_gpu...
  point_double_gpu ok
Testing point_add_gpu...
  point_add_gpu ok
Testing scalar_mul_gpu...
  scalar_mul_gpu ok


In [75]:
import time
import random

def benchmark_cpu_scalar_mul(ks):
    start = time.perf_counter()
    out = [scalar_multiply(k, G) for k in ks]
    end = time.perf_counter()
    return out, (end - start) * 1000

def benchmark_gpu_scalar_mul(ks):
    points = [G] * len(ks)

    start_event = cuda.Event()
    end_event = cuda.Event()

    start_wall = time.perf_counter()
    start_event.record()

    out = gpu_scalar_mul(points, ks)

    end_event.record()
    end_event.synchronize()
    end_wall = time.perf_counter()

    kernel_ms = start_event.time_till(end_event)
    total_ms = (end_wall - start_wall) * 1000

    return out, kernel_ms, total_ms

In [76]:
sizes = [1, 4, 16, 64, 256]

print(f"{'n':>6} | {'CPU ms':>12} | {'GPU kernel ms':>14} | {'GPU total ms':>12}")
print("-" * 55)

for n in sizes:
    ks = [random.randrange(1, 1 << 32) for _ in range(n)]

    cpu_out, cpu_ms = benchmark_cpu_scalar_mul(ks)
    gpu_out, gpu_kernel_ms, gpu_total_ms = benchmark_gpu_scalar_mul(ks)

    assert cpu_out == gpu_out

    print(f"{n:>6} | {cpu_ms:>12.3f} | {gpu_kernel_ms:>14.3f} | {gpu_total_ms:>12.3f}")

     n |       CPU ms |  GPU kernel ms | GPU total ms
-------------------------------------------------------
     1 |        5.232 |          2.107 |        2.147
     4 |       19.840 |          2.137 |        2.174
    16 |       80.254 |          2.387 |        2.417
    64 |      323.094 |          3.066 |        3.100
   256 |     1287.000 |          5.163 |        5.216
